# PCOS Synthetic Data Generation (TVAE)
Generates 2,500 synthetic rows and creates a master dataset.

In [14]:
!pip install -q sdv pandas numpy scikit-learn matplotlib

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

DATA_PATH = "pcos_clean.csv"

df = pd.read_csv(DATA_PATH)

print("Original Shape:", df.shape)
print("\nTarget Distribution:")
print(df["PCOS (Y/N)"].value_counts())

print("\nTarget Percentage:")
print(df["PCOS (Y/N)"].value_counts(normalize=True) * 100)

Original Shape: (541, 42)

Target Distribution:
PCOS (Y/N)
0    364
1    177
Name: count, dtype: int64

Target Percentage:
PCOS (Y/N)
0    67.28281
1    32.71719
Name: proportion, dtype: float64


In [16]:
TARGET = "PCOS (Y/N)"

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

train_df = pd.concat(
    [X_train, y_train],
    axis=1
)

test_df = pd.concat(
    [X_test, y_test],
    axis=1
)

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)

print("\nTraining target distribution:")
print(train_df[TARGET].value_counts())

print("\nTesting target distribution:")
print(test_df[TARGET].value_counts())

Training data: (432, 42)
Testing data: (109, 42)

Training target distribution:
PCOS (Y/N)
0    291
1    141
Name: count, dtype: int64

Testing target distribution:
PCOS (Y/N)
0    73
1    36
Name: count, dtype: int64


In [17]:
metadata = SingleTableMetadata()

metadata.detect_from_dataframe(train_df)

categorical_columns = [
    "PCOS (Y/N)",
    "Blood Group",
    "Cycle(R/I)",
    "Pregnant(Y/N)",
    "Weight gain(Y/N)",
    "hair growth(Y/N)",
    "Skin darkening (Y/N)",
    "Hair loss(Y/N)",
    "Pimples(Y/N)",
    "Fast food (Y/N)",
    "Reg.Exercise(Y/N)"
]

for column in categorical_columns:
    metadata.update_column(
        column_name=column,
        sdtype="categorical"
    )

print("Metadata created successfully.")

Metadata created successfully.


In [18]:
synthesizer = TVAESynthesizer(
    metadata=metadata,
    epochs=500,
    enforce_min_max_values=True,
    enforce_rounding=True,
    verbose=True
)

synthesizer.fit(train_df)

print("TVAE training completed.")

c:\Users\MARUTIRAJ\anaconda3\Lib\site-packages\sdv\single_table\base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
c:\Users\MARUTIRAJ\anaconda3\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Loss: +00.51: 100%|██████████| 500/500 [01:00<00:00,  8.32it/s]

TVAE training completed.


In [19]:
NUM_SYNTHETIC_ROWS = 1500

synthetic_df = synthesizer.sample(
    num_rows=NUM_SYNTHETIC_ROWS
)

print("Synthetic Shape:", synthetic_df.shape)

synthetic_df.to_csv(
    "synthetic_pcos.csv",
    index=False
)

print("Synthetic dataset saved successfully.")

Synthetic Shape: (1500, 42)
Synthetic dataset saved successfully.


In [20]:
real = train_df.copy()
synthetic = synthetic_df.copy()

numeric_columns = real.select_dtypes(
    include=np.number
).columns

comparison = []

for col in numeric_columns:

    real_mean = real[col].mean()
    synthetic_mean = synthetic[col].mean()

    real_std = real[col].std()
    synthetic_std = synthetic[col].std()

    comparison.append({
        "Column": col,
        "Real Mean": real_mean,
        "Synthetic Mean": synthetic_mean,
        "Mean Difference": abs(real_mean - synthetic_mean),
        "Real Std": real_std,
        "Synthetic Std": synthetic_std,
        "Std Difference": abs(real_std - synthetic_std)
    })

comparison_df = pd.DataFrame(comparison)

comparison_df = comparison_df.sort_values(
    "Mean Difference",
    ascending=False
)

display(comparison_df.head(20))

,Column,Real Mean,Synthetic Mean,Mean Difference,Real Std,Synthetic Std,Std Difference
13,I beta-HCG(mIU/mL),589.452884,147.106933,442.345952,3027.577337,143.699379,2883.877958
14,II beta-HCG(mIU/mL),224.408933,46.740159,177.668774,1607.640719,83.097232,1524.543487
24,Vit D3 (ng/mL),55.534125,29.428000,26.106125,387.272941,12.063194,375.209747
15,FSH(mIU/mL),17.052063,4.307689,12.744374,242.855306,2.359411,240.495895
17,FSH/LH,7.650600,2.704368,4.946232,67.837071,1.900644,65.936427
23,PRL(ng/mL),23.578310,18.939980,4.638330,13.709841,8.832995,4.876847
26,RBS(mg/dl),99.463426,95.559000,3.904426,15.448505,9.042885,6.405619
35,BP _Diastolic (mmHg),76.777778,79.780000,3.002222,5.839832,1.467323,4.372509
21,TSH (mIU/L),3.070991,1.786448,1.284543,3.964627,0.839710,3.124917
22,AMH(ng/mL),5.730655,4.453439,1.277216,5.991503,4.735475,1.256028


In [21]:
augmented_train_df = pd.concat(
    [train_df, synthetic_df],
    ignore_index=True
)

augmented_train_df = augmented_train_df.drop_duplicates()

augmented_train_df = augmented_train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

augmented_train_df.to_csv(
    "augmented_pcos_training.csv",
    index=False
)

print("Real training rows:", len(train_df))
print("Synthetic rows:", len(synthetic_df))
print("Augmented training rows:", len(augmented_train_df))

Real training rows: 432
Synthetic rows: 1500
Augmented training rows: 1932


In [22]:
X_train_real = train_df.drop(columns=[TARGET])
y_train_real = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

baseline_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

baseline_model.fit(
    X_train_real,
    y_train_real
)

baseline_pred = baseline_model.predict(X_test)
baseline_prob = baseline_model.predict_proba(X_test)[:, 1]

print("===== BASELINE MODEL =====")

print("Accuracy:",
      accuracy_score(y_test, baseline_pred))

print("Precision:",
      precision_score(y_test, baseline_pred))

print("Recall:",
      recall_score(y_test, baseline_pred))

print("F1:",
      f1_score(y_test, baseline_pred))

print("ROC-AUC:",
      roc_auc_score(y_test, baseline_prob))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        baseline_pred
    )
)

===== BASELINE MODEL =====
Accuracy: 0.926605504587156
Precision: 0.9666666666666667
Recall: 0.8055555555555556
F1: 0.8787878787878788
ROC-AUC: 0.9334094368340943

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.99      0.95        73
           1       0.97      0.81      0.88        36

    accuracy                           0.93       109
   macro avg       0.94      0.90      0.91       109
weighted avg       0.93      0.93      0.92       109



In [23]:
X_train_aug = augmented_train_df.drop(
    columns=[TARGET]
)

y_train_aug = augmented_train_df[TARGET]

augmented_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

augmented_model.fit(
    X_train_aug,
    y_train_aug
)

augmented_pred = augmented_model.predict(X_test)
augmented_prob = augmented_model.predict_proba(X_test)[:, 1]

print("===== SYNTHETIC-AUGMENTED MODEL =====")

print("Accuracy:",
      accuracy_score(y_test, augmented_pred))

print("Precision:",
      precision_score(y_test, augmented_pred))

print("Recall:",
      recall_score(y_test, augmented_pred))

print("F1:",
      f1_score(y_test, augmented_pred))

print("ROC-AUC:",
      roc_auc_score(y_test, augmented_prob))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        augmented_pred
    )
)

===== SYNTHETIC-AUGMENTED MODEL =====
Accuracy: 0.8899082568807339
Precision: 0.8529411764705882
Recall: 0.8055555555555556
F1: 0.8285714285714286
ROC-AUC: 0.9459665144596651

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92        73
           1       0.85      0.81      0.83        36

    accuracy                           0.89       109
   macro avg       0.88      0.87      0.87       109
weighted avg       0.89      0.89      0.89       109



In [24]:
results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Real Only": [
        accuracy_score(y_test, baseline_pred),
        precision_score(y_test, baseline_pred),
        recall_score(y_test, baseline_pred),
        f1_score(y_test, baseline_pred),
        roc_auc_score(y_test, baseline_prob)
    ],

    "Real + Synthetic": [
        accuracy_score(y_test, augmented_pred),
        precision_score(y_test, augmented_pred),
        recall_score(y_test, augmented_pred),
        f1_score(y_test, augmented_pred),
        roc_auc_score(y_test, augmented_prob)
    ]
})

results["Difference"] = (
    results["Real + Synthetic"]
    - results["Real Only"]
)

display(results)

,Metric,Real Only,Real + Synthetic,Difference
0,Accuracy,0.926606,0.889908,-0.036697
1,Precision,0.966667,0.852941,-0.113725
2,Recall,0.805556,0.805556,0.000000
3,F1,0.878788,0.828571,-0.050216
4,ROC-AUC,0.933409,0.945967,0.012557
